# 03. Cloudera AI에서 vLLM 실행

이 노트북에서는 **Cloudera AI** 환경의 구조를 이해하고,  
Beauty Fashion App을 Session 또는 Application으로 배포하는 방법을 설명합니다.

---

## Cloudera AI 구조

```
프로젝트
│
├── Session (세션)
│   └── Jupyter Notebook, Terminal 등 대화형 작업
│   └── CPU/GPU 리소스 직접 선택
│
├── Job (잡)
│   └── 배치 작업 스케줄링
│
└── Application (애플리케이션)
    └── 웹 서버를 외부에 공개 (Beauty Fashion App 배포)
    └── 포트 8002로 Session 접속 (Application 배포 시 cdsw-run.sh)
    └── cdsw-build.sh → 환경 구성
    └── cdsw-run.sh   → vLLM + FastAPI 시작
```

## Cell 1. cdsw-build.sh 역할 이해

CML에서 Application을 생성할 때 **자동으로 실행**되는 빌드 스크립트입니다.  
의존성 설치, 환경 구성, 프론트엔드 빌드 등을 수행합니다.

```bash
# cdsw-build.sh 내용 (우리 프로젝트)
#!/bin/bash
set -e  # 오류 발생 시 즉시 중단

echo "[1/2] Python 의존성 설치..."
pip install -r requirements/cloudera.txt

echo "[2/2] 빌드 완료!"
```

**CML에서 빌드가 실행되는 시점:**
1. Application 처음 생성할 때
2. "Rebuild" 버튼 클릭 시
3. 프로젝트 환경 재구성 시

## Cell 2. cdsw-run.sh 역할 이해

Application이 시작될 때 vLLM과 FastAPI를 순서대로 실행합니다.

```bash
# cdsw-run.sh 요약
# 1. vLLM 서버 백그라운드 시작 (포트 8001, CPU)
# 2. /v1/models 응답 대기
# 3. FastAPI 앱 시작 (포트 8002)
```

**포트 구성**: vLLM **8001** / FastAPI 앱 **8002**

## Cell 3. CPU에서 vLLM 실행 명령어

Cloudera AI Standard Runtime (2 vCPU / 4 GiB)에서 검증된 명령어입니다.

In [ ]:
cpu_command = """
# Cloudera AI Session (CPU, vllm-cpu 0.26.0)
python -m vllm.entrypoints.openai.api_server \\
  --model Qwen/Qwen2.5-0.5B-Instruct \\
  --host 0.0.0.0 \\
  --port 8001 \\
  --gpu-memory-utilization 0.35 \\
  --max-model-len 2048 \\
  --max-num-seqs 1

테스트: curl http://127.0.0.1:8001/v1/models
"""

print(cpu_command)
print("\n💡 CPU 환경 팁:")
print("  - pip install vllm-cpu (CUDA 빌드 아님)")
print("  - 4GiB RAM → 0.5B 모델 권장")
print("  - unset PIP_USER && export PIP_USER=0 (pip 오류 방지)")

## Cell 4. Application 생성 방법

```
프로젝트 → Applications → New Application
  - Name: beauty-fashion-ai
  - Script: cdsw-run.sh
  - Runtime: Python 3.11 Standard
  - Resource: 2 vCPU / 4 GiB (CPU)
  ↓
Create Application
  ↓
cdsw-build.sh (pip install) → cdsw-run.sh (vLLM + FastAPI)
  ↓
HTTPS URL에서 챗봇 사용
```

### Session에서 빠르게 테스트 (vLLM 이미 실행 중일 때)

터미널 2에서:

```bash
pip install -r requirements/base.txt
export VLLM_BASE_URL=http://127.0.0.1:8001/v1
export MODEL_NAME=Qwen/Qwen2.5-0.5B-Instruct
uvicorn app.main:app --host 0.0.0.0 --port 8002
```

## Cell 5. 환경 설정 (.env.cloudera.example)

In [ ]:
cloudera_env = {
    "설정 파일": ".env.cloudera.example",
    "VLLM_BASE_URL": "http://127.0.0.1:8001/v1",
    "MODEL_NAME": "Qwen/Qwen2.5-0.5B-Instruct",
    "VLLM_PORT": "8001",
    "APP_PORT": "8002",
    "MAX_TOKENS": "300",
    "GPU_MEMORY_UTILIZATION": "0.35",
    "MAX_MODEL_LEN": "2048",
}

print("=== Cloudera AI CPU 환경 설정 ===")
for key, val in cloudera_env.items():
    print(f"  {key}: {val}")

## 정리

**Cloudera AI에서 Beauty Fashion App 실행 흐름:**

```
1. Session에서 vllm-cpu 설치 및 vLLM 테스트 (완료)
       ↓
2-A. Session 터미널 2에서 uvicorn으로 앱 실행 (빠른 테스트)
   또는
2-B. Application 생성 → cdsw-run.sh로 vLLM + 앱 함께 배포 (권장)
       ↓
3. 브라우저에서 챗봇 UI 사용
```

---

다음: **`app/main.py`** 와 **`app/static/index.html`** 에서 실제 앱 코드를 확인하세요.